# ML Model Selection

Three supervised classifiers are trained and compared: **Decision Tree**, **Random Forest**, and **Logistic Regression**.  
Evaluation uses Precision, Recall, and F1-Score — accuracy alone is misleading on an imbalanced dataset.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score
warnings.filterwarnings("ignore")

df = pd.read_csv("../data/ai_resume_screening.csv").drop(columns=["github_activity"])

X = df.drop("shortlisted", axis=1)
y = df["shortlisted"].map({"Yes": 1, "No": 0})

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)
X_val,   X_test, y_val,   y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

education_order = {"High School": 0, "Bachelors": 1, "Masters": 2, "PhD": 3}
X_train_enc = X_train.copy(); X_val_enc = X_val.copy(); X_test_enc = X_test.copy()
for s in [X_train_enc, X_val_enc, X_test_enc]:
    s["education_level"] = s["education_level"].map(education_order)

classes = np.array([0, 1])
cw = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, cw))

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train_enc)
X_val_s   = scaler.transform(X_val_enc)
X_test_s  = scaler.transform(X_test_enc)

feature_names = X_train_enc.columns.tolist()
print("Setup complete.")
print(f"Features: {feature_names}")
print(f"Train: {X_train_s.shape[0]} | Val: {X_val_s.shape[0]} | Test: {X_test_s.shape[0]}")

## Model 1 — Decision Tree

Regularized with `max_depth`, `min_samples_split`, and `min_samples_leaf` to prevent overfitting.  
`criterion='entropy'` measures split quality via Information Gain.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=10,
    min_samples_split=40,
    min_samples_leaf=20,
    class_weight=class_weight_dict,
    random_state=42
)
dt.fit(X_train_s, y_train)

print("Decision Tree — Validation Set")
print(classification_report(y_val, dt.predict(X_val_s), target_names=["Not Shortlisted", "Shortlisted"]))

## Model 2 — Random Forest

Ensemble of 200 trees using bagging + random feature subsets to reduce variance.  
Also used for **feature importance** to generate candidate feedback.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=20,
    class_weight=class_weight_dict,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train_s, y_train)

print("Random Forest — Validation Set")
print(classification_report(y_val, rf.predict(X_val_s), target_names=["Not Shortlisted", "Shortlisted"]))

## Model 3 — Logistic Regression

Linear baseline using the sigmoid function for probabilistic binary classification.  
Regularization strength `C=1.0` balances fit and generalization.

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    solver="lbfgs",
    max_iter=300,
    C=1.0,
    class_weight=class_weight_dict,
    random_state=42
)
lr.fit(X_train_s, y_train)

print("Logistic Regression — Validation Set")
print(classification_report(y_val, lr.predict(X_val_s), target_names=["Not Shortlisted", "Shortlisted"]))

## Validation Comparison

In [ ]:
models = {"Decision Tree": dt, "Random Forest": rf, "Logistic Regression": lr}

rows = []
for name, model in models.items():
    preds = model.predict(X_val_s)
    rows.append({
        "Model": name,
        "Precision": round(precision_score(y_val, preds, zero_division=0), 4),
        "Recall":    round(recall_score(y_val, preds), 4),
        "F1-Score":  round(f1_score(y_val, preds), 4),
        "Accuracy":  round((preds == y_val).mean(), 4)
    })

comparison = pd.DataFrame(rows).set_index("Model")
comparison

## Final Test Set Evaluation

In [ ]:
print("Random Forest — Test Set")
print(classification_report(y_test, rf.predict(X_test_s), target_names=["Not Shortlisted", "Shortlisted"]))

print("Logistic Regression — Test Set")
print(classification_report(y_test, lr.predict(X_test_s), target_names=["Not Shortlisted", "Shortlisted"]))

## Feature Importance

Random Forest feature importances rank how much each feature contributed to predictions.  
Used to generate targeted improvement feedback for rejected candidates.

In [ ]:
import matplotlib.pyplot as plt

importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]
sorted_features = [feature_names[i] for i in indices]
sorted_importances = importances[indices]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(sorted_features, sorted_importances, color="steelblue", edgecolor="white")
ax.set_title("Random Forest — Feature Importances", fontsize=13, pad=10)
ax.set_ylabel("Importance")
ax.set_xlabel("Feature")
ax.tick_params(axis="x", rotation=20)
for bar, val in zip(bars, sorted_importances):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
            f"{val:.3f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

print("Feature importances:")
for f, imp in zip(sorted_features, sorted_importances):
    print(f"  {f:<25} {imp:.4f}")

## Feedback Generation

For a candidate predicted as *Not Shortlisted*, their feature values are compared against the median benchmarks of shortlisted candidates in the training set, sorted by feature importance.

In [ ]:
benchmarks = X_train_enc[y_train == 1].median()
edu_labels = {0: "High School", 1: "Bachelors", 2: "Masters", 3: "PhD"}

def generate_feedback(candidate: dict) -> str:
    """
    candidate: dict with keys matching feature_names.
    education_level should be provided as int (0=High School, 1=Bachelors, 2=Masters, 3=PhD).
    """
    row = pd.DataFrame([candidate])[feature_names]
    row_scaled = scaler.transform(row)
    pred = rf.predict(row_scaled)[0]

    if pred == 1:
        return "Strong profile — predicted to be shortlisted."

    importance_map = dict(zip(feature_names, rf.feature_importances_))
    lines = ["Profile is below the shortlisted threshold. Suggested improvements:\n"]
    for feat in sorted(feature_names, key=lambda f: importance_map[f], reverse=True):
        val = candidate[feat]
        bench = benchmarks[feat]
        if val < bench:
            if feat == "education_level":
                lines.append(f"  - education_level: {edu_labels.get(int(val))} -> consider {edu_labels.get(int(bench))}")
            else:
                lines.append(f"  - {feat}: {val} (benchmark: {bench:.1f})")
    return "\n".join(lines)


sample = {
    "years_experience":   2,
    "skills_match_score": 55.0,
    "education_level":    1,
    "project_count":      3,
    "resume_length":      300
}
print(generate_feedback(sample))